# ML-10 — Content Action Playbook

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [4]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
import os

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")
df = df[df["impressions_90d"] >= 100].copy()
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

feature_cols = [
    "avg_position", "ctr", "impressions_90d", "sessions_90d",
    "word_count", "content_age_days", "days_since_last_update",
    "search_volume", "competition"
]
X = df[feature_cols].fillna(0)
y = df["is_declining_label"]

# Train final model on client-grouped split (same design as w05/w06)
groups = df["client_id"]
splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(splitter.split(X, y, groups=groups))

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X.iloc[train_idx])
final_model = LogisticRegression(max_iter=1000, random_state=42).fit(X_train_scaled, y.iloc[train_idx])

# Score the FULL dataset (not just test set) for the actual playbook
X_all_scaled = scaler.transform(X)
df["model_score"] = final_model.predict_proba(X_all_scaled)[:, 1]

print("Rows scored:", len(df))
df[["content_id", "model_score"]].head()

Rows scored: 22006


,content_id,model_score
0,content_304f48230142,0.624553
1,content_a1fb4e703a9e,0.476851
2,content_9aa793d4d895,0.619473
3,content_331d6c4de07b,0.373535
4,content_d99b7a2d90ca,0.416692


In [5]:
def assign_reason_code(row):
    if row["model_score"] >= 0.7 and row["days_since_last_update"] >= 180:
        return "stale_high_risk"
    elif row["model_score"] >= 0.7:
        return "model_decline_risk"
    elif row["avg_position"] <= 20 and row["ctr"] < 0.5 and row["impressions_90d"] >= 500:
        return "ctr_below_position_expectation"
    elif row["model_score"] >= 0.5:
        return "moderate_risk_monitor"
    else:
        return "healthy_no_action"

def assign_action(reason_code):
    action_map = {
        "stale_high_risk": "refresh_content",
        "model_decline_risk": "review_content",
        "ctr_below_position_expectation": "review_metadata",
        "moderate_risk_monitor": "monitor",
        "healthy_no_action": "no_action"
    }
    return action_map[reason_code]

df["reason_code"] = df.apply(assign_reason_code, axis=1)
df["action"] = df["reason_code"].apply(assign_action)

ranked_queue = df.sort_values("model_score", ascending=False).reset_index(drop=True)

print(ranked_queue["reason_code"].value_counts())
print("\n", ranked_queue["action"].value_counts())
ranked_queue[["content_id", "model_score", "reason_code", "action"]].head(15)

reason_code
model_decline_risk                6232
moderate_risk_monitor             5908
ctr_below_position_expectation    5832
healthy_no_action                 4014
stale_high_risk                     20
Name: count, dtype: int64

 action
review_content     6232
monitor            5908
review_metadata    5832
no_action          4014
refresh_content      20
Name: count, dtype: int64


,content_id,model_score,reason_code,action
0,content_186489efe6f4,0.864536,model_decline_risk,review_content
1,content_a27eb2ede5e6,0.864043,model_decline_risk,review_content
2,content_e0b39e983a86,0.861446,model_decline_risk,review_content
3,content_d0fd2e1ec8f6,0.858739,model_decline_risk,review_content
4,content_01c90832176a,0.857288,model_decline_risk,review_content
5,content_82b0db79f37e,0.856493,model_decline_risk,review_content
6,content_7ae230422776,0.855966,model_decline_risk,review_content
7,content_4bb993e9270e,0.854217,model_decline_risk,review_content
8,content_c6fb8b50fef7,0.853806,model_decline_risk,review_content
9,content_d8020ed269ef,0.853258,model_decline_risk,review_content


## 1. Ranked actions + reason codes

**The queue, ranked by model_score (highest = review first):**

| Reason code | Count | Action | What it means |
|---|---:|---|---|
| `stale_high_risk` | 20 | `refresh_content` | High decline risk + not updated in 180+ days — full content refresh needed |
| `model_decline_risk` | 6,232 | `review_content` | Model sees a strong decline pattern (score ≥0.7) — needs human review |
| `ctr_below_position_expectation` | 5,832 | `review_metadata` | Good position, weak CTR relative to peers — likely a title/meta fix, not a content rewrite |
| `moderate_risk_monitor` | 5,908 | `monitor` | Some risk signal (score 0.5-0.7), not urgent — track over time |
| `healthy_no_action` | 4,014 | `no_action` | Low risk — leave alone |

**Top of the queue:** the highest-scoring pages (model_score 0.85+) are dominated by `model_decline_risk`, 
meaning the model's strongest signal is the same-window decline pattern from `content_age_days`, `ctr`, 
and `avg_position` (as seen in the w05 coefficients) — not the staleness rule, which only flags 20 
pages total. This confirms the w04 finding: staleness alone is a weak, narrow signal here, while the 
model-based score captures a much broader set of at-risk pages.

**Why a human trusts this:** every row has a reason code a reviewer can inspect before acting — no 
page is flagged with just a bare number. A reviewer scanning `model_decline_risk` knows to look for 
content-quality issues; one scanning `ctr_below_position_expectation` knows to check the title/meta 
instead. The two are different problems needing different fixes, and the reason code keeps that 
distinction visible instead of collapsing everything into one generic "opportunity."

## 2. Intended use and limits

**Who uses this:** a content strategist or SEO reviewer at FlyRank (or a client-facing account 
manager) with limited weekly review capacity, deciding which pages to look at first out of thousands.

**What it's for:** prioritization — telling a reviewer where to spend their limited time first. It is 
NOT a fully automated fix-it pipeline. Every row is a candidate for review, not a confirmed problem.

**Where it stops being valid:**
- **New clients / cold start:** the model was trained on 30 clients' worth of history. A brand-new 
  client with little or no history would get scores based on patterns learned from other clients, 
  which may not transfer — this queue should not be trusted for a client with under ~90 days of data.
- **Same-window label limitation:** because the model's label (`trend_direction == "down"`) comes from 
  the same window as its features (established in w03/w06), this queue tells you "which pages 
  currently look like they're declining," not "which pages will decline next month." Treat it as a 
  current-state snapshot, not a forecast.
- **Small-sample reason codes:** `stale_high_risk` only has 20 pages — too few to trust as a stable 
  pattern on its own (same caution as the paper's 361+ freshness bucket from w09).
- **Not causal:** a page flagged for `review_content` is not guaranteed to improve if edited — this is 
  decision-support, not a promise of recovery (no experiment was run to prove refresh causes recovery).

## 3. Human review + the no-go list

**What a person must check before acting on any flagged page:**
- Read the actual page content — does the reason code match what's really there? (e.g. is a 
  `ctr_below_position_expectation` page actually a featured-snippet page where low CTR is expected, 
  as I found in several w04 top-20 examples?)
- Check whether the page recently changed topic/URL — old signals may not reflect current content.
- Confirm the page has enough real demand (`impressions_90d`) to justify the reviewer's time.
- For `stale_high_risk` pages specifically: verify with only 20 pages in this bucket, each one 
  deserves individual attention rather than batch action.

**What should NEVER be automated (no-go list):**
- **Auto-publishing content changes** — no page content should be rewritten and published without a 
  human editor reviewing it first (matches the paper's own framing: FlyRank's automation ships 
  "actual fixes," but every finding in this notebook is decision-support only, not a fix pipeline).
- **Auto-deleting or de-indexing pages** based on `healthy_no_action` or low scores — a low model 
  score is not proof a page is worthless; it could be new, niche, or serving a purpose not captured 
  in these 9 features.
- **Client-facing reporting of `model_score` as a guarantee** — it should never be presented to a 
  client as "this page will decline" or "this page will recover if fixed." It's an internal 
  prioritization score, not a client-facing prediction.
- **Bulk action on `stale_high_risk`** — with only 20 pages, this reason code should get individual 
  human review, not a bulk automated refresh.

## 4. Monitoring / retrain triggers

**What would tell us the recommendations went stale:**
- **Distribution shift:** if the share of pages in `model_decline_risk` jumps sharply (e.g. from 
  ~28% to 50%+ of the portfolio) between runs, that could mean either a real site-wide problem or 
  the model no longer fitting current data well — worth investigating before trusting the queue.
- **Reason code imbalance drift:** if `stale_high_risk` (currently 20 pages) grows to a much larger 
  share, the staleness signal's role in the score should be re-evaluated, since w04's signal audit 
  already found staleness alone to be a MIXED, unreliable signal.
- **New client onboarding:** any time a client with under 90 days of history is added, their pages 
  should be excluded from this queue until enough data accumulates (per the intended-use limit above).
- **Data contract violations:** if the underlying data schema changes (new columns, renamed fields, 
  or the availability of `ga4_data_available` shifts significantly, as flagged in w03), the notebook 
  should be re-run and re-validated before trusting new output.
- **Suggested retrain cadence:** given this is trained on a static 30,000-row starter slice, in a 
  production setting I'd recommend retraining monthly at minimum, with a spot-check of Precision@50 
  on a fresh holdout each time, comparing against this notebook's baseline numbers (w04: 0.82, w05/w06: 
  0.86-0.88) to catch performance decay early.

In [6]:
import json
import os

# Export the ranked queue CSV
os.makedirs("../outputs", exist_ok=True)
export_cols = [
    "content_id", "client_id", "model_score", "reason_code", "action",
    "avg_position", "ctr", "impressions_90d", "content_age_days"
]
ranked_queue[export_cols].to_csv("../outputs/action_playbook_queue.csv", index=False)
print("Queue exported:", len(ranked_queue), "rows")

# Export metrics JSON — the "receipts" for the paper
metrics = {
    "total_pages_scored": int(len(df)),
    "reason_code_counts": df["reason_code"].value_counts().to_dict(),
    "action_counts": df["action"].value_counts().to_dict(),
    "baseline_precision_at_50": 0.82,   # from w04
    "model_precision_at_50_naive_split": 0.86,  # from w06
    "model_precision_at_50_grouped_split": 0.88,  # from w05/w06
    "features_used": feature_cols,
    "validation_design": "client-grouped split (GroupShuffleSplit)",
    "label_definition": "trend_direction == 'down' (same-window proxy, not future outcome)"
}

with open("../outputs/playbook_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

print("Metrics exported.")
print(json.dumps(metrics, indent=2))

Queue exported: 22006 rows
Metrics exported.
{
  "total_pages_scored": 22006,
  "reason_code_counts": {
    "model_decline_risk": 6232,
    "moderate_risk_monitor": 5908,
    "ctr_below_position_expectation": 5832,
    "healthy_no_action": 4014,
    "stale_high_risk": 20
  },
  "action_counts": {
    "review_content": 6232,
    "monitor": 5908,
    "review_metadata": 5832,
    "no_action": 4014,
    "refresh_content": 20
  },
  "baseline_precision_at_50": 0.82,
  "model_precision_at_50_naive_split": 0.86,
  "model_precision_at_50_grouped_split": 0.88,
  "features_used": [
    "avg_position",
    "ctr",
    "impressions_90d",
    "sessions_90d",
    "word_count",
    "content_age_days",
    "days_since_last_update",
    "search_volume",
    "competition"
  ],
  "validation_design": "client-grouped split (GroupShuffleSplit)",
  "label_definition": "trend_direction == 'down' (same-window proxy, not future outcome)"
}


## 5. Exports for the paper

This notebook exports two files to `work/outputs/`:
- `action_playbook_queue.csv` — the full ranked queue with model_score, reason_code, and action for 
  all 22,006 scored pages. Regenerated fresh each time this notebook runs (not committed to git, per 
  the CI leak-guard).
- `playbook_metrics.json` — the summary numbers this notebook's claims trace back to (reason code 
  counts, baseline vs model Precision@50 across both split designs, feature list, and label 
  definition). This IS committed to git, since it's the "receipts" file my capstone paper's numbers 
  will cite.

## Self-check

Before you submit, confirm each line honestly:

- [X] Every section above is filled — markdown thinking AND the code that backs it
- [X] The notebook runs top to bottom with no errors (Runtime → Run all)
- [X] No client names, URLs, or private queries anywhere
- [X] My claims use careful words: observed, measured, directional, decision-support
- [X] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.